# parameter recovery

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["OMP_NUM_THREADS"] = "1"
import seaborn as sns 
from scipy.io import loadmat
import warnings
warnings.filterwarnings("ignore")

In [8]:
outputFolderName = "param_recovery_2_reward_distribution"
inputFolderName = r"D:\Nill\data\BART\0_0_new_IED\context_modeling\param_recovery_0_data"


if not os.path.exists(outputFolderName):
    os.makedirs(outputFolderName)

In [9]:
matFiles = [f for f in os.listdir(inputFolderName) if f.endswith(".mat")]
nPatients = len(matFiles)

data_by_color = {
    1: [],  # yellow
    2: [],  # orange
    3: []   # red
}

for pt in range(nPatients):

    fileName = matFiles[pt]

    ptID = os.path.splitext(fileName)[0]
    ptID = ptID.replace("_TDdataParamRecovery", "")

    print(f"processing pt {pt+1}/{nPatients}: {ptID}")

    matFile = os.path.join(inputFolderName, fileName)
    mat = loadmat(matFile, struct_as_record=False, squeeze_me=True)

    TDdataParamRecovery = mat["TDdataParamRecovery"]

    result = np.array(TDdataParamRecovery.result, dtype=str)
    reward = TDdataParamRecovery.Reward
    isControl = TDdataParamRecovery.is_control
    trial_type = TDdataParamRecovery.trial_type
    points = TDdataParamRecovery.points

    # loop over trials
    for i in range(len(trial_type)):
        color = trial_type[i]

        if color not in [1, 2, 3]:
            continue

        trial_result = str(result[i]).strip().lower()

        # control + banked -> use actual reward
        if isControl[i] == 1 and trial_result == "banked":
            data_by_color[color].append(reward[i])

        # non-control + popped -> use points as hypothetical reward
        elif isControl[i] == 0 and trial_result == "popped":
            data_by_color[color].append(points[i])



processing pt 1/71: 201810
processing pt 2/71: 201811
processing pt 3/71: 201901
processing pt 4/71: 201902r
processing pt 5/71: 201902
processing pt 6/71: 201903
processing pt 7/71: 201905
processing pt 8/71: 201909
processing pt 9/71: 201910
processing pt 10/71: 201911
processing pt 11/71: 201914
processing pt 12/71: 201915
processing pt 13/71: 202001
processing pt 14/71: 202002
processing pt 15/71: 202003
processing pt 16/71: 202004
processing pt 17/71: 202005
processing pt 18/71: 202006u
processing pt 19/71: 202007
processing pt 20/71: 202008
processing pt 21/71: 202011
processing pt 22/71: 202014
processing pt 23/71: 202015
processing pt 24/71: 202016
processing pt 25/71: 202105
processing pt 26/71: 202107
processing pt 27/71: 202110
processing pt 28/71: 202114
processing pt 29/71: 202117
processing pt 30/71: 202118
processing pt 31/71: 202201
processing pt 32/71: 202202
processing pt 33/71: 202205
processing pt 34/71: 202207
processing pt 35/71: 202208
processing pt 36/71: 202209

In [10]:
summary = []

color_map = {
    1: "yellow",
    2: "orange",
    3: "red"
}

for color in [1, 2, 3]:
    values = np.array(data_by_color[color])

    mean_val = np.mean(values)
    std_val = np.std(values)

    summary.append({
        "color": color_map[color],
        "mean_reward": mean_val,
        "std_reward": std_val,
        "n_trials": len(values)
    })


df_summary = pd.DataFrame(summary)

# save file
output_path = os.path.join(outputFolderName, "reward_summary_by_color.csv")
df_summary.to_csv(output_path, index=False)

print("\nSaved to:", output_path)
print(df_summary)


Saved to: param_recovery_2_reward_distribution\reward_summary_by_color.csv
    color  mean_reward  std_reward  n_trials
0  yellow   279.420577   86.663573      1769
1  orange   196.374276   56.208364      1726
2     red   120.371961   33.070574      3003


this is the results if we don't count popped trials:


    color  mean_reward  std_reward  n_trials
0  yellow   292.853971   90.814231      1171
1  orange   201.853345   60.410843      1166
2     red   100.093937   29.328169      1171

# debug

In [11]:
fields = [f for f in dir(TDdataParamRecovery) if not f.startswith('_')]
print(fields)

['Reward', 'inflate_time', 'is_control', 'points', 'pointsMinusReward', 'result', 'scoreVec', 'trial_type']
